# Phase 7B — run the Streamlit churn prediction app in Colab

This notebook launches the Phase 7B app using the trusted Phase 7A model artifact created by Notebook 12. It does **not** retrain, tune, calibrate, or evaluate the model.

**Run Notebook 12 first in the same Colab runtime.** The model artifact is intentionally not stored in Git. This notebook copies that local artifact into a separate app checkout and starts Streamlit through Colab's port proxy. Do not upload customer data that you are not authorized to process.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

SOURCE_ARTIFACT = Path('/content/customer-churn-prediction/models/customer_churn_model.joblib')
APP_ROOT = Path('/content/customer-churn-prediction-phase-7b')
BRANCH = 'feature/phase-7b-streamlit-app'

if not SOURCE_ARTIFACT.is_file():
    raise FileNotFoundError(
        'Phase 7A artifact not found. Run Notebook 12 in this same Colab runtime first; '
        'do not commit or upload the .joblib file to GitHub.'
    )

if APP_ROOT.exists():
    shutil.rmtree(APP_ROOT)
subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH,
     'https://github.com/adhishmodi/customer-churn-prediction.git', str(APP_ROOT)],
    check=True,
)
(APP_ROOT / 'models').mkdir(exist_ok=True)
shutil.copy2(SOURCE_ARTIFACT, APP_ROOT / 'models' / 'customer_churn_model.joblib')
print('App source:', APP_ROOT)
print('Trusted model artifact copied into the app runtime; it remains outside Git.')


In [ ]:
!pip install "streamlit>=1.30"


In [ ]:
from google.colab import output
import time

if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()

log_path = APP_ROOT / 'streamlit.log'
log_handle = open(log_path, 'w', encoding='utf-8')
streamlit_process = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app.py',
     '--server.port=8501', '--server.address=0.0.0.0',
     '--server.headless=true', '--server.enableCORS=false',
     '--server.enableXsrfProtection=false'],
    cwd=APP_ROOT,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
)
time.sleep(5)
if streamlit_process.poll() is not None:
    log_handle.flush()
    raise RuntimeError('Streamlit did not start. Inspect /content/customer-churn-prediction-phase-7b/streamlit.log')

print('Streamlit is running. Use the embedded app window below.')
output.serve_kernel_port_as_window(8501)


## Stop the app

Run this cell when you finish testing. Restarting the Colab runtime also stops the app.

In [ ]:
if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    print('Streamlit process stopped.')
else:
    print('No running Streamlit process found.')
